# Three-Dimensional `Receiver`

This notebook provides an empirical demonstration of the expected behavior of a three-dimensional `Receiver`. Its purpose is to make the module's spatial specialization observable in a simple setting in which it can be studied in isolation, rather than to exhaustively validate every implementation detail.

The capability examined is adaptive refinement of the input space. The `Receiver` starts with no prior distinction between different input regions. As stimuli are received, frequently visited regions progressively split into smaller cuboids, while less frequently visited regions remain comparatively coarse. The resulting partition therefore allocates more spatial resolution where inputs occur more often.

## Conceptual Overview

Within Jacinta, a `Receiver` is responsible for receiving input stimuli: the continuous coordinates presented to the processing system. In this three-dimensional example, an input stimulus is a triplet of real numbers in a bounded cuboidal region. The module does not choose where those stimuli occur. Instead, it observes externally generated inputs and progressively specializes its representation of the regions in which they are received.

The interaction loop is therefore:

1. An external process generates a `ReceiverSample`.
2. `forward()` locates the leaf containing that input and delegates output sampling to its associated `Transmitter`.
3. `backward()` records the visit and eventually refines the corresponding input region when enough evidence has accumulated.

At initialization, the `Receiver` has no evidence that any part of the input space requires more resolution than another. Its tree therefore consists of one cuboid spanning the complete domain. Spatial distinctions emerge only from the frequency with which different regions are visited.

### How the input space is represented

Representing a continuous input space with uniform high resolution would require a fixed discretization, including fine partitions in regions that may rarely or never be observed. The `Receiver` instead specializes `NDSpace`, Jacinta's tree-based representation of continuous space. In three dimensions, every leaf of the tree represents an axis-aligned cuboid, and each incoming coordinate is routed to the leaf whose bounds contain it.

Spatial specialization is driven by visits. Every leaf maintains a hit counter, and once its configured threshold is reached, it attempts to split at its midpoint. A split replaces the original cuboid with eight equal children. Subsequent inputs are then distinguished at this finer spatial resolution.

The refinement process is local. Receiving many stimuli in one part of the domain does not force unrelated regions to split. Consequently, regions visited more frequently accumulate hits faster and tend to become smaller, while regions receiving little input can remain represented by larger cuboids. The tree therefore adapts its spatial resolution to the empirical input distribution rather than imposing the same resolution everywhere.

Each receiver region is associated with a `Transmitter`, providing the output-side state that can eventually specialize for that input context. When a receiver leaf splits, each child inherits a copy of the parent's current `Transmitter`. This preserves the state already associated with the broader region while allowing the newly distinguished contexts to evolve independently in subsequent interactions.

### Scope of the demonstration

The sampling function used below is stationary: the relative frequency with which each input region is visited does not change during the run. This isolates the input-side specialization behavior and makes the resulting partition easy to interpret. The notebook therefore demonstrates adaptive spatial refinement *from sequential input exposure*, but it does not test how the module responds when the input distribution itself changes over time.

The associated `Transmitter` is deliberately prevented from refining in this experiment, and every interaction uses zero feedback. Its purpose here is only to satisfy the `Receiver` interaction interface, keeping the experiment focused on how the input partition evolves from visit frequency.

In [1]:
import math
import random
from collections.abc import Callable

import matplotlib.pyplot as plt
from IPython.display import Image
from matplotlib import colormaps
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from PIL import Image as PILImage

from jacinta.processor.evaluator import ZScoreEvaluator
from jacinta.processor.receiver import Receiver, ReceiverSample
from jacinta.processor.transmitter import Transmitter
from jacinta.utils.scheduler import ConstantScheduler

## Configuration

The parameters below are chosen to make the relevant behavior visible within a short, reproducible run. They provide enough interactions for several levels of local refinement to occur, but they should not be interpreted as general-purpose defaults.

The input domain is the half-open cube $[-10,10)\times[-10,10)\times[-10,10)$. The half-open convention follows `NDSpace`: the lower bound of each dimension is included and the upper bound is excluded.

### Spatial refinement

Refinement is controlled by `HITS_RATE = 1000`: after a leaf has received one thousand input stimuli, it attempts to split at its midpoint. A lower threshold would produce finer partitions sooner, based on fewer observations; a higher threshold would keep the representation coarse for longer.

`MIN_WIDTH = 0.001` provides the stopping condition for refinement. A split is rejected if any child would be narrower than this value along any dimension. The limit is intentionally much smaller than the cuboids expected during this experiment, so refinement is determined primarily by visit frequency rather than by reaching the minimum width. `MAX_DEPTH` can therefore remain `None`, since the minimum-width constraint already places a finite bound on the tree depth.

### Scheduler and reproducibility

Hit rate is a depth-dependent parameter in the `Receiver` API. This notebook wraps `HITS_RATE` in a `ConstantScheduler`, so the same setting is used at every depth. Other schedulers can make refinement more or less selective at different depths.

`SEED = 42` makes the externally generated input sequence reproducible, `N_STEPS = 40000` supplies enough interactions for several levels of refinement, and `N_PLOTS = 5` records approximately evenly spaced snapshots of the evolving partition.

In [2]:
MIN_X, MAX_X = -10.0, 10.0
MIN_Y, MAX_Y = -10.0, 10.0
MIN_Z, MAX_Z = -10.0, 10.0
MIN_P, MAX_P = 0.0, 1.0

# hyperparameters
HITS_RATE = 1000
MIN_WIDTH = 0.001
MAX_DEPTH = None
SEED = 42

# simulation parameters
N_STEPS = 40000
N_PLOTS = 5

## Visualization Utilities

The `Receiver` stores its spatial representation implicitly as a tree rather than as a fixed discretization of the input domain. The following helpers expose that structure solely for inspection; they do not participate in routing, sampling, or refinement.

`get_leaves()` recursively traverses the tree and returns its current leaves. Since every leaf represents one active input region, their bounds describe the complete spatial partition maintained by the `Receiver`.

A complete three-dimensional partition cannot be inspected clearly from a single static projection because the projections of many cuboids overlap visually. `plot_receiver()` therefore renders two-dimensional slices through the partition as axis-aligned rectangles embedded in the three-dimensional domain. Color deliberately carries no additional information: the visualization is intended to show only the spatial specialization itself. Rectangle boundaries therefore correspond directly to divisions in the receiver tree. Finer subdivisions within a slice indicate regions that have been refined more strongly, whereas larger rectangles indicate areas that remain comparatively coarse. `animate_receiver()` sweeps those slices through the domain so that the complete three-dimensional partition can be inspected over time.

`eval_sampling_function()` and `plot_sampling_function()` evaluate the external sampling function over the same domain. The three spatial axes represent the input coordinates, while relative sampling probability is represented only by color. `animate_sampling_function()` sweeps orthogonal slices through the distribution, while `animate_receiver()` places a sampling slice beside corresponding slices of the receiver partition. This makes it possible to compare where input stimuli occur more frequently with where the `Receiver` has allocated additional spatial resolution.

In [3]:
def eval_sampling_function(
    sampling_function: Callable[[float, float, float], float],
    x: float,
    y: float,
    z: float,
) -> float:
    """
    Safely evaluate a sampling function for the given values.

    Args:
        sampling_function (Callable[[float, float, float], float]): The sampling
            function to evaluate.
        x (float): The first value to evaluate the sampling function at.
        y (float): The second value to evaluate the sampling function at.
        z (float): The third value to evaluate the sampling function at.

    Returns:
        float: The result of the sampling function evaluation.
    """
    try:
        p = sampling_function(x, y, z)
        if not math.isfinite(p):
            p = math.nan
        p = max(min(p, MAX_P), MIN_P)
    except (ArithmeticError, ValueError):
        p = math.nan
    return p

In [4]:
def plot_sampling_function(
    ax: plt.Axes,
    sampling_function: Callable[[float, float, float], float],
    slice_ax: str,
    slice_val: float,
    n_points: int = 100,
    elev: float = 30.0,
    azim: float = 30.0,
    colorbar: bool = True,
) -> None:
    """
    Plot a slice of a sampling function.

    Args:
        ax (plt.Axes): The axes to plot on.
        sampling_function (Callable[[float, float, float], float]): The sampling
            function to plot.
        slice_ax (str): The axis to slice along.
        slice_val (float): The value of the sliced axis.
        n_points (int): The number of points to plot per dimension.
            Defaults to 100.
        elev (float): The elevation of the viewing angle.
            Defaults to 30.0.
        azim (float): The azimuth of the viewing angle.
            Defaults to 30.0.
        colorbar (bool): Whether to plot the color bar.
            Defaults to True.
    """
    # evaluate the sampling function
    if slice_ax == "x":
        y_step = (MAX_Y - MIN_Y) / (n_points - 1)
        z_step = (MAX_Z - MIN_Z) / (n_points - 1)
        y_vals = [MIN_Y + idx * y_step for idx in range(n_points)]
        z_vals = [MIN_Z + idx * z_step for idx in range(n_points)]
        x_grid = [slice_val for z_val in z_vals for y_val in y_vals]
        y_grid = [y_val for z_val in z_vals for y_val in y_vals]
        z_grid = [z_val for z_val in z_vals for y_val in y_vals]
        p_grid = [
            eval_sampling_function(sampling_function, slice_val, y_val, z_val)
            for z_val in z_vals
            for y_val in y_vals
        ]
    elif slice_ax == "y":
        x_step = (MAX_X - MIN_X) / (n_points - 1)
        z_step = (MAX_Z - MIN_Z) / (n_points - 1)
        x_vals = [MIN_X + idx * x_step for idx in range(n_points)]
        z_vals = [MIN_Z + idx * z_step for idx in range(n_points)]
        x_grid = [x_val for z_val in z_vals for x_val in x_vals]
        y_grid = [slice_val for z_val in z_vals for x_val in x_vals]
        z_grid = [z_val for z_val in z_vals for x_val in x_vals]
        p_grid = [
            eval_sampling_function(sampling_function, x_val, slice_val, z_val)
            for z_val in z_vals
            for x_val in x_vals
        ]
    elif slice_ax == "z":
        x_step = (MAX_X - MIN_X) / (n_points - 1)
        y_step = (MAX_Y - MIN_Y) / (n_points - 1)
        x_vals = [MIN_X + idx * x_step for idx in range(n_points)]
        y_vals = [MIN_Y + idx * y_step for idx in range(n_points)]
        x_grid = [x_val for y_val in y_vals for x_val in x_vals]
        y_grid = [y_val for y_val in y_vals for x_val in x_vals]
        z_grid = [slice_val for y_val in y_vals for x_val in x_vals]
        p_grid = [
            eval_sampling_function(sampling_function, x_val, y_val, slice_val)
            for y_val in y_vals
            for x_val in x_vals
        ]
    # normalize the sampling function values for the color map
    normalize = Normalize(vmin=MIN_P, vmax=MAX_P)
    colormap = colormaps["summer"]
    # build the mesh triangles
    triangles = []
    for row in range(n_points - 1):
        for col in range(n_points - 1):
            # build the mesh cell
            top_left = row * n_points + col
            top_right = top_left + 1
            bottom_left = (row + 1) * n_points + col
            bottom_right = bottom_left + 1
            # split the cell into two triangles
            triangles.append([top_left, top_right, bottom_right])
            triangles.append([top_left, bottom_right, bottom_left])
    # get the average sampling function value of each triangle
    p_vals = [sum(p_grid[idx] for idx in triangle) / 3 for triangle in triangles]
    # map each sampling function value to a color
    facecolors = [colormap(normalize(p_val)) for p_val in p_vals]
    # plot the sampling function
    surface = ax.plot_trisurf(
        x_grid,
        y_grid,
        z_grid,
        triangles=triangles,
        antialiased=False,
    )
    surface.set_facecolors(facecolors)
    if colorbar:
        # set the height and bottom of the color bar
        colorbar_height = 0.9
        colorbar_bottom = (1.0 - colorbar_height) / 2
        # plot the color bar
        scalar_mappable = ScalarMappable(norm=normalize, cmap=colormap)
        scalar_mappable.set_array([])
        colorbar_ax = ax.inset_axes([1.02, colorbar_bottom, 0.04, colorbar_height])
        colorbar = ax.figure.colorbar(scalar_mappable, cax=colorbar_ax)
        colorbar.set_label("p")
        colorbar.ax.yaxis.set_label_position("left")
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlim(MIN_X, MAX_X)
    ax.set_ylim(MIN_Y, MAX_Y)
    ax.set_zlim(MIN_Z, MAX_Z)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_title(
        f"Sampling Function ({slice_ax}={slice_val:.4f}, elev={elev}, azim={azim})"
    )
    ax.grid(visible=True)
    return

In [5]:
def get_leaves(receiver: Receiver) -> list[Receiver]:
    """
    Get the receiver leaves.

    Args:
        receiver (Receiver): The receiver to get the leaves of.

    Returns:
        list[Receiver]: A list containing the receiver leaves.
    """
    # check if the receiver is a leaf
    if receiver.is_leaf:
        leaves = [receiver]
    # otherwise, recursively get its leaves
    else:
        leaves = []
        for child in receiver.children:
            child_leaves = get_leaves(child)
            leaves.extend(child_leaves)
    return leaves

In [6]:
def plot_receiver(
    ax: plt.Axes,
    receiver: Receiver,
    slice_ax: str,
    slice_val: float,
    elev: float = 30.0,
    azim: float = 30.0,
) -> None:
    """
    Plot a slice of the spatial specialization of a receiver.

    Args:
        ax (plt.Axes): The axes to plot on.
        receiver (Receiver): The receiver to plot.
        slice_ax (str): The axis to slice along.
        slice_val (float): The value of the sliced axis.
        elev (float): The elevation of the viewing angle.
            Defaults to 30.0.
        azim (float): The azimuth of the viewing angle.
            Defaults to 30.0.
    """
    # get the receiver leaves
    leaves = get_leaves(receiver)
    # sort the leaves by their lower bounds
    leaves.sort(
        key=lambda leaf: (leaf.bounds[0][0], leaf.bounds[1][0], leaf.bounds[2][0])
    )
    # get the sliced leaves
    rectangles = []
    for leaf in leaves:
        x_lower, x_upper = leaf.bounds[0]
        y_lower, y_upper = leaf.bounds[1]
        z_lower, z_upper = leaf.bounds[2]
        if slice_ax == "x":
            # skip leaves that do not intersect the slice
            if not x_lower <= slice_val < x_upper:
                continue
            # build the rectangular intersection of the leaf with the slice
            vertices = [
                (slice_val, y_lower, z_lower),
                (slice_val, y_upper, z_lower),
                (slice_val, y_upper, z_upper),
                (slice_val, y_lower, z_upper),
            ]
        elif slice_ax == "y":
            if not y_lower <= slice_val < y_upper:
                continue
            vertices = [
                (x_lower, slice_val, z_lower),
                (x_upper, slice_val, z_lower),
                (x_upper, slice_val, z_upper),
                (x_lower, slice_val, z_upper),
            ]
        elif slice_ax == "z":
            if not z_lower <= slice_val < z_upper:
                continue
            vertices = [
                (x_lower, y_lower, slice_val),
                (x_upper, y_lower, slice_val),
                (x_upper, y_upper, slice_val),
                (x_lower, y_upper, slice_val),
            ]
        rectangles.append(vertices)
    # plot the receiver
    collection = Poly3DCollection(
        rectangles,
        facecolors="C0",
        edgecolors="black",
        antialiased=False,
    )
    ax.add_collection3d(collection)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlim(MIN_X, MAX_X)
    ax.set_ylim(MIN_Y, MAX_Y)
    ax.set_zlim(MIN_Z, MAX_Z)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_title(
        "Receiver Spatial Specialization\n"
        f"({slice_ax}={slice_val:.4f}, elev={elev}, azim={azim})"
    )
    ax.grid(visible=True)
    return

In [7]:
def create_gif(
    fig: plt.Figure,
    update: Callable[[int], None],
    filepath: str,
    n_frames: int = 100,
    interval: int = 100,
) -> Image:
    """
    Create a GIF animation.

    Args:
        fig (plt.Figure): The figure to animate.
        update (Callable[[int], None]): The function to update each frame.
        filepath (str): The path where the animation is saved.
        n_frames (int): The number of frames to animate.
            Defaults to 100.
        interval (int): The interval between frames in milliseconds.
            Defaults to 100.

    Returns:
        Image: The GIF animation.
    """
    # render the animation frames
    frames = []
    for frame in range(n_frames):
        update(frame)
        fig.canvas.draw()
        image = PILImage.frombuffer(
            "RGBA",
            fig.canvas.get_width_height(),
            fig.canvas.buffer_rgba(),
        )
        frames.append(image.convert("RGB"))
    # create the animation (in memory)
    frames[0].save(
        filepath,
        format="GIF",
        save_all=True,
        append_images=frames[1:],
        duration=interval,
        loop=0,
    )
    plt.close(fig)
    animation = Image(filename=filepath)
    return animation

In [8]:
def animate_sampling_function(
    sampling_function: Callable[[float, float, float], float],
    filepath: str,
    n_points: int = 100,
    n_frames: int = 100,
    interval: int = 100,
    elev: float = 30.0,
    azim: float = 30.0,
    colorbar: bool = True,
) -> Image:
    """
    Animate slices of a sampling function.

    Args:
        sampling_function (Callable[[float, float, float], float]): The sampling
            function to animate.
        filepath (str): The path where the animation is saved.
        n_points (int): The number of points to plot per dimension.
            Defaults to 100.
        n_frames (int): The number of frames to animate.
            Defaults to 100.
        interval (int): The interval between frames in milliseconds.
            Defaults to 100.
        elev (float): The elevation of the viewing angle.
            Defaults to 30.0.
        azim (float): The azimuth of the viewing angle.
            Defaults to 30.0.
        colorbar (bool): Whether to plot the color bar.
            Defaults to True.

    Returns:
        Image: The sampling function animation.
    """
    # get the slices
    x_step = (MAX_X - MIN_X) / (n_frames - 1)
    y_step = (MAX_Y - MIN_Y) / (n_frames - 1)
    z_step = (MAX_Z - MIN_Z) / (n_frames - 1)
    x_vals = [MIN_X + idx * x_step for idx in range(n_frames)]
    y_vals = [MIN_Y + idx * y_step for idx in range(n_frames)]
    z_vals = [MIN_Z + idx * z_step for idx in range(n_frames)]

    def update(frame: int) -> None:
        """
        Update the animation frame.

        Args:
            frame (int): The frame index.
        """
        # clear the previous frame
        for axis in ax:
            axis.clear()
        # plot the slices
        plot_sampling_function(
            ax[0],
            sampling_function,
            slice_ax="x",
            slice_val=x_vals[frame],
            n_points=n_points,
            elev=elev,
            azim=azim,
            colorbar=False,
        )
        plot_sampling_function(
            ax[1],
            sampling_function,
            slice_ax="y",
            slice_val=y_vals[frame],
            n_points=n_points,
            elev=elev,
            azim=azim,
            colorbar=False,
        )
        plot_sampling_function(
            ax[2],
            sampling_function,
            slice_ax="z",
            slice_val=z_vals[frame],
            n_points=n_points,
            elev=elev,
            azim=azim,
            colorbar=colorbar,
        )
        # adjust the aspect ratio of each plot
        for axis in ax:
            axis.set_box_aspect((1, 1, 1), zoom=0.95)
        return

    # adjust the layout
    fig, ax = plt.subplots(ncols=3, figsize=(20, 7), subplot_kw={"projection": "3d"})
    update(0)
    fig.tight_layout()
    # create the animation
    animation = create_gif(fig, update, filepath, n_frames=n_frames, interval=interval)
    return animation

In [9]:
def animate_receiver(
    receiver: Receiver,
    sampling_function: Callable[[float, float, float], float],
    filepath: str,
    n_points: int = 100,
    n_frames: int = 100,
    interval: int = 100,
    elev: float = 30.0,
    azim: float = 30.0,
    colorbar: bool = True,
) -> Image:
    """
    Animate slices of a receiver and its sampling function.

    Args:
        receiver (Receiver): The receiver to animate.
        sampling_function (Callable[[float, float, float], float]): The sampling
            function to animate.
        filepath (str): The path where the animation is saved.
        n_points (int): The number of points to plot per dimension.
            Defaults to 100.
        n_frames (int): The number of frames to animate.
            Defaults to 100.
        interval (int): The interval between frames in milliseconds.
            Defaults to 100.
        elev (float): The elevation of the viewing angle.
            Defaults to 30.0.
        azim (float): The azimuth of the viewing angle.
            Defaults to 30.0.
        colorbar (bool): Whether to plot the color bar.
            Defaults to True.

    Returns:
        Image: The receiver animation.
    """
    # get the slices
    y_step = (MAX_Y - MIN_Y) / (n_frames - 1)
    z_step = (MAX_Z - MIN_Z) / (n_frames - 1)
    y_vals = [MIN_Y + idx * y_step for idx in range(n_frames)]
    z_vals = [MIN_Z + idx * z_step for idx in range(n_frames)]

    def update(frame: int) -> None:
        """
        Update the animation frame.

        Args:
            frame (int): The frame index.
        """
        # clear the previous frame
        for axis in ax:
            axis.clear()
        # plot the slices
        plot_sampling_function(
            ax[0],
            sampling_function,
            slice_ax="z",
            slice_val=z_vals[frame],
            n_points=n_points,
            elev=elev,
            azim=azim,
            colorbar=colorbar,
        )
        plot_receiver(
            ax[1],
            receiver,
            slice_ax="z",
            slice_val=z_vals[frame],
            elev=elev,
            azim=azim,
        )
        plot_receiver(
            ax[2],
            receiver,
            slice_ax="y",
            slice_val=y_vals[frame],
            elev=elev,
            azim=azim,
        )
        # adjust the aspect ratio of each plot
        for axis in ax:
            axis.set_box_aspect((1, 1, 1), zoom=0.95)
        return

    # adjust the layout
    fig, ax = plt.subplots(ncols=3, figsize=(20, 7), subplot_kw={"projection": "3d"})
    update(0)
    fig.tight_layout()
    # create the animation
    animation = create_gif(fig, update, filepath, n_frames=n_frames, interval=interval)
    return animation

## Sampling Function

The `Receiver` does not choose its own inputs, so this playground needs an external process that generates input stimuli. In this controlled setting, that process is represented by a stationary sampling function: coordinates are drawn repeatedly from the same underlying distribution. This makes the evolution of the receiver's spatial specialization easier to interpret.

The sampling function combines three smooth peaks with different relative magnitudes and spatial extents. Its global maximum is approximately 1.0 near $(x,y,z)=(-4.5,4.0,-2.5)$; lower local maxima of approximately 0.65 and 0.35 occur near $(1.5,-3.0,5.0)$ and $(6.0,1.5,-5.5)$, respectively. The distribution is deliberately nonuniform and multimodal. A uniform distribution would tend to refine the input space evenly, whereas spatially separated regions with different sampling frequencies make it possible to observe whether the `Receiver` allocates finer spatial resolution where stimuli occur more often.

The animated slices are shown for explanation, but they are not available to the `Receiver`. The module receives neither the sampling function nor the relative probability associated with each coordinate. At each interaction it observes only the input stimulus that was sampled. The resulting spatial specialization is therefore driven entirely by the frequency with which different regions of the input space are encountered.

In [10]:
def sampling_function(x: float, y: float, z: float) -> float:
    """
    Sampling function.

    Args:
        x (float): The first value to evaluate the sampling function at.
        y (float): The second value to evaluate the sampling function at.
        z (float): The third value to evaluate the sampling function at.

    Returns:
        float: The relative probability of sampling the point.
    """
    p = (
        1.00
        * math.exp(
            -0.5
            * (((x + 4.5) / 1.2) ** 2 + ((y - 4.0) / 1.7) ** 2 + ((z + 2.5) / 1.4) ** 2)
        )
        + 0.65
        * math.exp(
            -0.5
            * (((x - 1.5) / 2.0) ** 2 + ((y + 3.0) / 1.1) ** 2 + ((z - 5.0) / 1.8) ** 2)
        )
        + 0.35
        * math.exp(
            -0.5
            * (((x - 6.0) / 1.0) ** 2 + ((y - 1.5) / 1.5) ** 2 + ((z + 5.5) / 1.2) ** 2)
        )
    )
    return p


# plot the sampling function
animation = animate_sampling_function(
    sampling_function,
    filepath=(
        "../../../../assets/jacinta/processor/receiver"
        "/three_dimensional_receiver/sampling_function.gif"
    ),
    n_frames=50,
    elev=25.0,
    colorbar=True,
)

![Sampling Function](../../../../assets/jacinta/processor/receiver/three_dimensional_receiver/sampling_function.gif)

In [11]:
def get_sampling_function_max(
    sampling_function: Callable[[float, float, float], float],
    n_points: int = 100,
) -> float:
    """
    Approximate the maximum of a sampling function.

    Args:
        sampling_function (Callable[[float, float, float], float]): The sampling
            function to evaluate.
        n_points (int): The number of points to evaluate per dimension.
            Defaults to 100.

    Returns:
        float: The approximate maximum of the sampling function.
    """
    # evaluate the sampling function
    x_step = (MAX_X - MIN_X) / (n_points - 1)
    y_step = (MAX_Y - MIN_Y) / (n_points - 1)
    z_step = (MAX_Z - MIN_Z) / (n_points - 1)
    x_vals = [MIN_X + idx * x_step for idx in range(n_points)]
    y_vals = [MIN_Y + idx * y_step for idx in range(n_points)]
    z_vals = [MIN_Z + idx * z_step for idx in range(n_points)]
    p_vals = [
        eval_sampling_function(sampling_function, x_val, y_val, z_val)
        for z_val in z_vals
        for y_val in y_vals
        for x_val in x_vals
    ]
    # get the sampling function maximum
    max_p = max(p_vals)
    return max_p

In [12]:
def sample(
    sampling_function: Callable[[float, float, float], float],
    max_p: float,
    rng: random.Random,
) -> ReceiverSample:
    """
    Sample from a sampling function.

    Args:
        sampling_function (Callable[[float, float, float], float]): The sampling
            function.
        max_p (float): The upper bound used for rejection sampling.
        rng (random.Random): The random number generator used for sampling.

    Returns:
        ReceiverSample: The sampled input.
    """
    # sample using rejection sampling
    while True:
        x = rng.uniform(MIN_X, MAX_X)
        y = rng.uniform(MIN_Y, MAX_Y)
        z = rng.uniform(MIN_Z, MAX_Z)
        p = rng.uniform(0.0, max_p)
        if p <= eval_sampling_function(sampling_function, x, y, z):
            break
    # generate the receiver sample
    rsample = ReceiverSample(
        coordinates=(x, y, z),
    )
    return rsample

## Simulation

The `Receiver` is now exposed to repeated input stimuli drawn from the sampling function. It begins as a single leaf spanning the complete region, so every coordinate initially belongs to the same input region. The random seed makes the sampled sequence reproducible.

Each iteration executes the complete interaction cycle:

1. `sample()` draws an input coordinate from the stationary sampling distribution.
2. `forward(bias=0.0)` locates the receiver leaf containing that coordinate and samples from its associated `Transmitter`.
3. `backward()` records the visit and refines the corresponding receiver region whenever its hit threshold is reached.

The associated `Transmitter` cannot refine in this experiment, and the feedback passed to `backward()` is always zero, so it plays no active role beyond satisfying the interaction interface. The changes visible in the receiver animations are driven only by the frequency with which different parts of the input space are visited. Each rectangular boundary in a slice marks the intersection of a division between receiver leaves with that plane, so smaller rectangles indicate greater spatial specialization.

### Reading the snapshots

The first snapshot contains a single region because only one interaction has occurred and the root has not yet reached its split threshold.

By step 10,001, the receiver has undergone several coarse refinements across the domain. The strongest specialization has already begun around the dominant peak near $(-4.5,4.0,-2.5)$, while the regions associated with the secondary sampling modes have also begun to separate.

By step 20,001, the relationship between sampling frequency and spatial resolution is substantially clearer. Multiple successive splits have accumulated around the dominant peak, producing finer spatial subdivisions in its surrounding region. Additional refinement is visible around the secondary peak near $(1.5,-3.0,5.0)$ and the weaker peak near $(6.0,1.5,-5.5)$, while low-frequency regions remain represented by much larger cuboids.

By step 30,000, this trend continues rather than introducing a new behavior. Further splits concentrate around the same three high-frequency regions, increasing the contrast between the locally refined areas and the comparatively coarse remainder of the domain.

By the final snapshot, the partition is clearly nonuniform. The finest spatial specialization appears around the dominant peak near $(-4.5,4.0,-2.5)$, followed by the secondary regions near $(1.5,-3.0,5.0)$ and $(6.0,1.5,-5.5)$. Large parts of the domain in which the sampling probability is low remain comparatively coarse. The sequence therefore shows the intended behavior of the `Receiver`: spatial resolution progressively concentrates where input stimuli are encountered more frequently.

In [13]:
# initialize a 1D Transmitter
transmitter = Transmitter(
    bounds=((-10.0, 10.0),),
    evaluator=ZScoreEvaluator(0.001, 0.001),
    bias_scale_scheduler=ConstantScheduler(value=10.0),
    learning_rate_scheduler=ConstantScheduler(value=0.001),
    hits_rate_scheduler=ConstantScheduler(value=1000),
    min_width=1,
    max_depth=0,
    seed=42,
)

In [14]:
# initialize a 3D Receiver
receiver = Receiver(
    bounds=((MIN_X, MAX_X), (MIN_Y, MAX_Y), (MIN_Z, MAX_Z)),
    transmitter=transmitter,
    hits_rate_scheduler=ConstantScheduler(value=HITS_RATE),
    min_width=MIN_WIDTH,
    max_depth=MAX_DEPTH,
)

In [15]:
# approximate the sampling function maximum
max_p = get_sampling_function_max(sampling_function)

# initialize the random number generator
rng = random.Random(SEED)

# set the receiver plot steps
plot_steps = (
    {round(idx * (N_STEPS - 1) / (N_PLOTS - 1)) for idx in range(N_PLOTS)}
    if N_PLOTS > 1
    else {N_STEPS - 1}
)

# run simulation
for step in range(N_STEPS):
    print(f"Step {step + 1}/{N_STEPS}", end="\r")
    rsample = sample(sampling_function, max_p, rng)
    tsample = receiver.forward(rsample, bias=0.0)
    receiver.backward(rsample, tsample, feedback=0.0)
    # plot the receiver
    if step in plot_steps:
        animation = animate_receiver(
            receiver,
            sampling_function,
            filepath=(
                "../../../../assets/jacinta/processor/receiver"
                f"/three_dimensional_receiver/step_{step + 1}.gif"
            ),
            n_frames=50,
            elev=25.0,
            colorbar=True,
        )

Step 1/40000

![Simulation (Step 1)](../../../../assets/jacinta/processor/receiver/three_dimensional_receiver/step_1.gif)

Step 10001/40000

![Simulation (Step 10001)](../../../../assets/jacinta/processor/receiver/three_dimensional_receiver/step_10001.gif)

Step 20001/40000

![Simulation (Step 20001)](../../../../assets/jacinta/processor/receiver/three_dimensional_receiver/step_20001.gif)

Step 30000/40000

![Simulation (Step 30000)](../../../../assets/jacinta/processor/receiver/three_dimensional_receiver/step_30000.gif)

Step 40000/40000

![Simulation (Step 40000)](../../../../assets/jacinta/processor/receiver/three_dimensional_receiver/step_40000.gif)

## Conclusions

This playground provides an empirical view of how a three-dimensional `Receiver` adaptively specializes a continuous input space. It begins with a single region spanning the complete domain and receives only a sequence of input coordinates. From that interaction history, it progressively refines the spatial partition according to where stimuli occur.

Refinement is driven by visits. Regions that are encountered more frequently accumulate hits faster and therefore undergo more successive splits, producing smaller cuboids and finer spatial resolution. Less frequently visited regions remain comparatively coarse. In the reproduced run, this process concentrates refinement around the strongest modes of the sampling distribution, with lower-probability parts of the domain retaining a coarser representation.

The resulting partition does not require prior knowledge of the input distribution. The `Receiver` observes only the coordinates presented to it, while the distribution of those observations determines where additional spatial resolution is created.

A standalone `Receiver` provides the input-side spatial specialization used by Jacinta. When combined with `Transmitter` inside `Processor`, each specialized input region provides a distinct context for output behavior. Examined in isolation, the module exhibits its intended behavior: it adaptively allocates finer spatial resolution to the regions of the input space that are encountered more frequently.